In [1]:
# Imports
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "normal"

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# srun -w ruapehu -c 20 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha
# ssh -L 8889:localhost:8890 ls985@ruapehu

In [2]:
def analyse_gpu_metrics(csv_file: Path) -> tuple[list[float], list[float]]:
    """Analyse the GPU metrics from a .csv file and return a pandas DataFrame."""
    # Read the .csv file into a pandas DataFrame
    gpu_dataframe = pd.read_csv(
        csv_file,
        header=None,
        names=["index", "gpu_util", "mem_total", "mem_used", "mem_free", "timestamp"],
        skipfooter=1,
        engine="python",
    )
    # Create a new column for the memory used fraction
    gpu_dataframe["mem_used_frac"] = (
        gpu_dataframe["mem_used"] / gpu_dataframe["mem_total"]
    )
    # Drop the columns that are not needed
    gpu_dataframe = gpu_dataframe.drop(columns=["mem_free", "mem_total"])
    # Convert the `timestamp` column to datetime
    gpu_dataframe["timestamp"] = pd.to_datetime(gpu_dataframe["timestamp"])
    # Convert `timestamp` to seconds
    gpu_dataframe["timestamp"] = gpu_dataframe["timestamp"].astype(int) / 10**9
    # Filter-out rows where `mem_used` is lower than 100 (MB)
    gpu_dataframe = gpu_dataframe[gpu_dataframe["mem_used"] > 100]  # noqa: PLR2004
    # gpu_dataframe = gpu_dataframe[
    #     gpu_dataframe["mem_used"] > gpu_dataframe["mem_used"].min()
    # ]
    # Shift the timestamp to start from 0
    gpu_dataframe["timestamp"] = (
        gpu_dataframe["timestamp"] - gpu_dataframe["timestamp"].min()
    )
    # Rename colum `index` to `GPU Index`
    gpu_dataframe = gpu_dataframe.rename(columns={"index": "GPU Index"})
    # Compute AUC for `gpu_util` and `mem_used` columns over `timestamp`
    gpu_util_gpus: list[float] = []
    total_idle_times: list[float] = []
    for gpu_id in gpu_dataframe["GPU Index"].unique():
        single_gpu_dataframe = gpu_dataframe[
            gpu_dataframe["GPU Index"] == gpu_id
        ].reindex()
        gpu_util = single_gpu_dataframe["gpu_util"]
        single_gpu_dataframe["timestamp"] = (
            single_gpu_dataframe["timestamp"] - single_gpu_dataframe["timestamp"].min()
        )
        single_gpu_dataframe["timestamp_diff"] = single_gpu_dataframe[
            "timestamp"
        ].diff()
        timestamp = single_gpu_dataframe["timestamp"]
        gpu_util_gpus.append(np.trapz(gpu_util, timestamp) / timestamp.max())
        # Get the difference between subsequent timestamps
        zero_util_single_gpu_dataframe = single_gpu_dataframe[
            single_gpu_dataframe["gpu_util"] <= 0
        ].reindex()
        zero_util_timestamp_diff = zero_util_single_gpu_dataframe["timestamp_diff"]
        total_idle_times.append(np.sum(zero_util_timestamp_diff))
    return gpu_util_gpus, total_idle_times

In [3]:
# openimage LB
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-7edf601f-*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-a49c066*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-c4562*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

61.653466466086776 0.4421714891992808
26.445666948954266 0.29769037170269375
29.798196604023946 8.984532968676072
70.67144436306424 22.363607407318877


In [4]:
# openimage RR
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-018c4e06-*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-92f7*.csv"):
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-00792*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

68.09515824585446 0.8157838271531214
17.898249804973602 4.144364715995826
18.304166509493857 2.605843574207085
114.4236665169398 11.649860513644585


In [5]:
# google speech LB
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-1c8*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-dc349*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-fa5d*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

19.357715948417013 0.32853302281511
76.60500041643779 16.451894619208495
7.384364173052628 0.49575035101939174
93.15266675419278 14.919173366846332


In [6]:
# google speech RR
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-47de*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-8bd8*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-97f6*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

18.95003904732958 1.3491625181124158
68.02666687965393 16.80827604640273
6.910902539796581 0.8271956893900543
114.99322242206998 25.391965882456294


In [7]:
# shakespeare LB
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-4434*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-660ff*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-ebcde*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

45.1667285795748 3.2217162249825484
13.87933357556661 9.728193470543545
16.04590412767653 4.162630495527612
52.79944430457221 8.68103375054676


In [8]:
# shakespeare RR
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-e7c7*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-1a575*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-674bd*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

51.05580866866611 1.6690834584114729
9.3836669921875 8.412768786204879
17.818943150439818 2.7163214750625344
42.682888719770645 6.444497643608892


In [9]:
# reddit LB
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-9e2d*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-65838*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-fd316*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

60.70162722727761 0.5453312380698571
44.78000028928121 12.170439132929138
26.817515812673516 3.428859240082005
133.79733350541858 13.499599656418246


In [10]:
# reddit RR
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
gpu_util_a40: list[float] = []
idle_time_a40: list[float] = []
gpu_util_2080: list[float] = []
idle_time_2080: list[float] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-df6a*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-573fe*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-6d9c9094*.csv"):
    # Read the .csv file into a pandas DataFrame
    _gpu_util, _idle_time = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    if len(_gpu_util) == 1:
        gpu_util_a40.append(_gpu_util[0])
        idle_time_a40.append(_idle_time[0])
    else:
        gpu_util_2080.extend(_gpu_util)
        idle_time_2080.extend(_idle_time)
print(np.mean(gpu_util_a40), np.std(gpu_util_a40))
print(np.mean(idle_time_a40), np.std(idle_time_a40))
print(np.mean(gpu_util_2080), np.std(gpu_util_2080))
print(np.mean(idle_time_2080), np.std(idle_time_2080))

66.29363619404917 0.9001011132337509
27.409666299819946 0.7692206237735771
19.071412639814213 3.671353122969729
183.15333324008517 34.34158578310459
